# 05 · Flood disruption and network resilience

**Question:** Which routes and residents lose access when low streets close?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Give Astra before/after maps and ask it to distinguish longer travel from complete disconnection.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Threshold inundation is a screening model
This static elevation threshold ignores rainfall, drainage, flow and hydraulic connectivity.
An edge closes when either endpoint is below the threshold. That is conservative but can miss low points between endpoints.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200); nodes,graph=network()
x,y=nodes.T
z=1000+.003*x+.002*y-4*np.exp(-((x-1100)/250)**2)
water_level=1001.0; closed=z<water_level
wet_graph={u:[(v,w) for v,w in edges if not(closed[u] or closed[v])] for u,edges in graph.items()}
idx,connector=snap(c['xy'],nodes); clinic=120
assert not closed[clinic], 'Choose a facility above the water threshold.'
base=(shortest(graph,[clinic])[idx]+connector)/80
flood=(shortest(wet_graph,[clinic])[idx]+connector)/80
reachable=np.isfinite(flood)
assert (flood[reachable]>=base[reachable]-1e-9).all()
unserved=int(pop[~reachable].sum())
print('Residents unreachable:',unserved,'of',pop.sum())
if reachable.any(): print('Mean travel among reachable residents:',np.average(flood[reachable],weights=pop[reachable]))


In [ ]:
fig,ax=plt.subplots()
for u,edges in graph.items():
    for v,w in edges:
        if u<v: ax.plot(*nodes[[u,v]].T,color='#e45756' if closed[u] or closed[v] else '#b9c9ce',lw=1)
ax.scatter(*c['xy'][reachable].T,c='#087f8c',s=15,label='Reachable')
ax.scatter(*c['xy'][~reachable].T,c='#e45756',marker='x',label='Disconnected')
ax.scatter(*nodes[clinic],marker='*',s=160,c='gold',label='Clinic')
map_axes(ax,'Static flood screening'); ax.legend(); plt.show()
export_json('05_disruption.json',dict(synthetic=True,water_level=water_level,unreachable_population=unserved,total_population=int(pop.sum())))


## Teaching lab: Audit disconnection
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 05_disruption.json and the flood-screening map. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Report the fraction of residents disconnected and explain why a finite mean among reachable residents is not a whole-population mean.

**Implementation brief — review the runnable extension below:**
> Add a sweep over water thresholds while retaining a dry facility. Plot unreachable population separately from reachable-only travel time. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Removing more edges cannot restore reachability; unreachable residents remain in the total population denominator.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Flood thresholds, edge sampling and alternative facilities

Compare endpoint-only closures with endpoint-plus-midpoint sampling over a range of water levels. Two alternative clinics are available only while dry. The plots keep unreachable people separate from travel time among reachable people. More sampling can reveal low road segments; it does not turn this screening equation into a hydraulic model.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Flood thresholds, edge sampling and alternative facilities', ['Terrain + streets + clinics', 'Sample endpoints AND midpoint', 'Disconnected people by level', 'Screening is not hydraulics'])
plt.show()


In [ ]:
ex_c=city(); ex_pop=allocate(ex_c['capacity'],1200); ex_nodes,ex_graph=network(); ex_idx,ex_connector=snap(ex_c['xy'],ex_nodes)
def ex_height(points):
    x,y=np.asarray(points).T; return 1000+.003*x+.002*y-4*np.exp(-((x-1100)/250)**2)
ex_levels=np.arange(998.,1005.1,1.); ex_counts={'Endpoints':[],'With midpoint':[]}; ex_means=[]
for level in ex_levels:
    ex_clinics=[node for node in [110,120] if ex_height(ex_nodes[[node]])[0]>=level]
    for method in ex_counts:
        graph={u:[] for u in ex_graph}
        for u,edges in ex_graph.items():
            for v,w in edges:
                samples=np.array([ex_nodes[u],ex_nodes[v]])
                if method=='With midpoint': samples=np.vstack([samples,(ex_nodes[u]+ex_nodes[v])/2])
                if ex_height(samples).min()>=level: graph[u].append((v,w))
        times=(shortest(graph,ex_clinics)[ex_idx]+ex_connector)/80
        reachable=np.isfinite(times); ex_counts[method].append(int(ex_pop[~reachable].sum()))
        if method=='With midpoint': ex_means.append(float(np.average(times[reachable],weights=ex_pop[reachable])) if ex_pop[reachable].sum() else None)
        if method=='With midpoint' and level==1001.: ex_example=(graph,times,ex_clinics)
assert (np.diff(ex_counts['With midpoint'])>=0).all()
assert (np.array(ex_counts['With midpoint'])>=ex_counts['Endpoints']).all()
fig,axes=plt.subplots(1,3,figsize=(14,4))
for method,counts in ex_counts.items(): axes[0].plot(ex_levels,counts,marker='o',label=method)
axes[0].set(title='Disconnected people, all residents counted',xlabel='Screening water level (m)',ylabel='Residents'); axes[0].legend(fontsize=8)
axes[1].plot(ex_levels,[np.nan if v is None else v for v in ex_means],marker='o'); axes[1].set(title='Conditional mean: reachable people only',xlabel='Water level (m)',ylabel='Walking min')
graph,times,clinics=ex_example
for u,edges in graph.items():
    for v,w in edges:
        if u<v: axes[2].plot(*ex_nodes[[u,v]].T,c='#bacad0',lw=.6)
axes[2].scatter(*ex_c['xy'].T,c=np.isfinite(times),cmap='coolwarm_r',s=20)
axes[2].scatter(*ex_nodes[clinics].T,c='gold',marker='*',s=100); map_axes(axes[2],'1001 m: blue reachable, red disconnected')
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(levels_m=ex_levels.tolist(),unreachable_population=ex_counts,reachable_mean_min=ex_means,total_population=1200)
extension_expected=dict(max_unreachable=max(ex_counts['With midpoint']),midpoint_never_improves_access=True)
extension_task='Report max_unreachable across midpoint scenarios and midpoint_never_improves_access, a boolean describing whether stricter closure sampling can restore a path in this fixed graph.'


In [ ]:
extension_prompt,extension_reference=export_investigation('05',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 05_extension_prompt.json and optionally 05_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `05_extension_prompt.json` and `05_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('05_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
